# 12.03 · Async agents and concurrency

**Module 12 · Production & Deployment** · Part 2 (Pro)

| | |
|---|---|
| ⏱️ Estimated time | 2–3 h |
| 📋 Prerequisites | Lessons 12.01–12.02, Module 02 (streaming) and Project 09 (the support desk). |
| 🔑 Keys | none (optional: `GROQ_API_KEY` for the 💸 concurrent-calls probe) |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-16 |

> **How to use this notebook:** read each Markdown cell, then run the code cell below it with **Shift + Enter**, top to bottom.
> Cell markers: **🧪** offline (scripted model, no network, free) · **🌐** network, no tokens · **💸** calls a model (uses your free-tier quota).
> Without a key, 💸 cells print `⏭️ Skipped` and everything else still runs, so **Kernel → Restart Kernel and Run All Cells…** always completes.

## Learning objectives

1. **Explain** how one event loop serves many runs, and what blocks it.
2. **Find and fix** blocking code inside async agents (`time.sleep`, sync clients) with `await` or `asyncio.to_thread`.
3. **Know** where sync code runs in an async server (a thread pool), and the limit that pool sets.
4. **Run** many agent runs at once with a concurrency limit and a bounded queue, and measure queue time against the latency budget of 12.01.
5. **Apply** timeouts and cancellation without leaving work half done.
6. **Size** concurrency from the provider's rate limit with Little's law.

## 📚 Official reading (verified 2026-10-16)

| Page | Read it for |
|---|---|
| [Concurrency and async / await](https://fastapi.tiangolo.com/async/) (FastAPI) | When to write `async def`, and what happens to plain `def` endpoints (a thread pool) |
| [asyncio tasks](https://docs.python.org/3/library/asyncio-task.html) | `gather`, `TaskGroup`, `timeout`, `to_thread`, cancellation |
| [asyncio synchronization](https://docs.python.org/3/library/asyncio-sync.html) | `Semaphore` for concurrency limits |
| [LangGraph: use the graph API (async)](https://docs.langchain.com/oss/python/langgraph/use-graph-api) | `ainvoke` / `astream` and async nodes |
| [Runnable interface](https://reference.langchain.com/python/langchain-core/runnables/base/Runnable) | `abatch(..., config={"max_concurrency": n})` |

---
## Step 0 · Install and set up

In [ ]:
%pip install --quiet "langchain>=1.4.4,<2" "langchain-core>=1.6.9,<2" "langchain-groq>=1.1.3,<2" "langgraph>=1.2.14,<2"

**You should now see** *"Note: you may need to restart the kernel to use updated packages."*, possibly below a few lines from pip. IPython prints that note after **every** `%pip`; restart only if a package was **upgraded** after you had already imported it (Lesson 00.05). `%pip` always installs into **this kernel's** environment; `!pip` might not (Lesson 00.02).

In [ ]:
from importlib import metadata

for package in ("langchain", "langgraph", "langchain-groq",):
    print(f"{package:<22} {metadata.version(package)}")

In [ ]:
# Setup 1: Find the repository root (Jupyter starts in modules/12-production-and-deployment/).
import importlib.util
import os
import sys
from pathlib import Path

# Optional: set LC_MASTERY_ROOT to your clone if Jupyter starts outside it (no paths are hardcoded).
_override = os.environ.get("LC_MASTERY_ROOT", "").strip()
REPO_ROOT = next(
    (p for p in ([Path(_override).expanduser()] if _override else [Path.cwd(), *Path.cwd().parents])
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the course repo. Start Jupyter inside your clone (or set LC_MASTERY_ROOT to it).")

# Setup 2: Import `mastery` from THIS clone's src/, like pytest does (pythonpath = ["src"]), and check
# that the env's own install (00.02 guide, Step 5) points here too: terminal commands use that one.
SRC_DIR = (REPO_ROOT / "src").resolve()
installed_spec = importlib.util.find_spec("mastery")
if sys.path[0] != str(SRC_DIR):
    sys.path.insert(0, str(SRC_DIR))
import mastery

print("Repo root:", REPO_ROOT)
print("mastery  :", mastery.__version__, "from", mastery.__file__)
installed_at = installed_spec.origin if installed_spec else None
if not Path(mastery.__file__).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This kernel had already imported mastery from another folder: restart the kernel, run from the top.")
elif installed_at is None or not Path(installed_at).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This env's own mastery install does not use this clone's src/:", installed_at or "(not installed)")
    print("   This notebook uses this clone's src/ anyway, but terminal commands (python -m mastery...) do not.")
    print('   Fix once (Git Bash, course env active, root of THIS clone): python -m pip install -e ".[dev]"')

In [ ]:
KEYS_TO_CHECK = ('GROQ_API_KEY', 'LANGSMITH_API_KEY')
LANGSMITH_PROJECT = 'm12-async-concurrency'
# Setup 3: Load .env, check keys WITHOUT printing them, read model IDs from config.
import os

from mastery.config import env_status, is_set, load_env, load_settings

dotenv_path = load_env()
settings = load_settings(load_dotenv_file=False)   # .env was just loaded.
HAVE_GROQ_KEY = is_set("GROQ_API_KEY")
HAVE_LANGSMITH_KEY = is_set("LANGSMITH_API_KEY")
os.environ.setdefault("LANGSMITH_PROJECT", LANGSMITH_PROJECT)   # m12-<lesson>: one tracing project per lesson

print(".env      :", dotenv_path or "not found (fine if keys are set in your shell)")
for name, status in env_status(*KEYS_TO_CHECK).items():
    print(f"{name:<18}: {status}")
print("project   :", os.environ["LANGSMITH_PROJECT"])
print("fast      :", settings.model_fast)
print("reasoning :", settings.model_reasoning)
if not HAVE_GROQ_KEY:
    print("\n⚠️ No GROQ_API_KEY: 💸 cells will print ⏭️ Skipped; 🧪 cells still teach the concept.")
if not HAVE_LANGSMITH_KEY:
    print("⚠️ No LANGSMITH_API_KEY: 🌐 LangSmith cells will print ⏭️ Skipped; offline twins run instead.")

In [ ]:
# Setup 4: Silence known, harmless noise so real problems stand out.
import os
import warnings
from importlib.metadata import version

os.environ.setdefault("ANONYMIZED_TELEMETRY", "False")          # Chroma (Project 09 LIVE FAQ index): no usage telemetry.
# langsmith warns once when a Client is created without a key; offline cells never send anything.
warnings.filterwarnings("ignore", message="API key must be provided when using hosted LangSmith API")
# Same as Lesson 02.04: older langchain-core 1.6.x Pydantic serializer warning about the injected runtime.
# Only older releases raise it; on current ones this filter is skipped.
if tuple(int("".join(c for c in x if c.isdigit()) or 0) for x in version("langchain-core").split(".")[:3]) < (1, 6, 11):
    warnings.filterwarnings("ignore", message="Pydantic serializer warnings", category=UserWarning)
# fastembed (Project 09 LIVE FAQ index) imports tqdm.auto; without ipywidgets it warns once. Progress bars still work.
warnings.filterwarnings("ignore", message="IProgress not found")

In [ ]:
def table(rows, columns):
    """Print dict rows as an aligned text table (one comparison cell per Part)."""
    widths = {c: max(len(c), *(len(str(r.get(c, ""))) for r in rows)) for c in columns}
    print("  ".join(c.ljust(widths[c]) for c in columns))
    for r in rows:
        print("  ".join(str(r.get(c, "")).ljust(widths[c]) for c in columns))

---
## Part 1 · One loop, many waits

An agent run spends almost all its time **waiting**: for the model's first token, for an HTTP tool, for the database. An `asyncio` event loop runs other work while one task waits, so one process can serve many runs at once, but only if every wait is an `await`. A plain `time.sleep` (or any blocking call) stops the whole loop.

> Jupyter already runs an event loop, so notebook cells can use `await` directly. In a script you'd wrap the same code in `asyncio.run(main())`.

In [ ]:
# 🧪 Offline: ten "requests" that each wait 0.2 s, three ways.
import asyncio
import time

async def blocking_wait():
    time.sleep(0.2)              # blocks the loop: nothing else runs

async def async_wait():
    await asyncio.sleep(0.2)     # yields to the loop while waiting

def sync_wait():
    time.sleep(0.2)              # sync library code (e.g. an SDK without async)

async def timed(label, make):
    start = time.perf_counter()
    await asyncio.gather(*(make() for _ in range(10)))
    return {"style": label, "wall_s": round(time.perf_counter() - start, 2)}

rows = [
    await timed("async def + time.sleep", blocking_wait),
    await timed("async def + await asyncio.sleep", async_wait),
    await timed("sync code via asyncio.to_thread", lambda: asyncio.to_thread(sync_wait)),
]
table(rows, ["style", "wall_s"])

**You should now see** about 2.0 s for the blocking version and about 0.2 s for the other two.

### What just happened?

- `async def` alone doesn't make code concurrent: the blocking version took 10 × 0.2 s.
- `await asyncio.sleep` (or an async HTTP client) lets the waits overlap.
- Sync code you can't change goes to a worker thread with `asyncio.to_thread`: it overlaps too, at the cost of a thread per call.

---
## Part 2 · The same trap inside an agent

Tools are where blocking code usually hides: an `async def` tool that calls `requests.get` or `time.sleep`. Five concurrent runs of a one-tool agent, with three versions of the same 0.3 s tool:

In [ ]:
# 🧪 Offline: 5 concurrent agent runs; the tool waits 0.3 s.
from langchain.agents import create_agent
from langchain_core.messages import AIMessage
from langchain_core.tools import tool

from mastery.testing import ScriptedChatModel

@tool
async def status_blocking(service: str) -> str:
    """Check a service's status."""
    time.sleep(0.3)
    return f"{service}: ok"

@tool
async def status_async(service: str) -> str:
    """Check a service's status."""
    await asyncio.sleep(0.3)
    return f"{service}: ok"

@tool
def status_sync(service: str) -> str:
    """Check a service's status."""
    time.sleep(0.3)              # a plain def tool: LangChain runs it in a worker thread under ainvoke
    return f"{service}: ok"

def agent_with(status_tool):
    script = [AIMessage("", tool_calls=[{"name": status_tool.name, "args": {"service": "api"}, "id": "c1"}]),
              "All systems normal."]
    return create_agent(ScriptedChatModel(responses=script), tools=[status_tool])

async def five_runs(status_tool):
    start = time.perf_counter()
    await asyncio.gather(*(agent_with(status_tool).ainvoke({"messages": [{"role": "user", "content": "status?"}]})
                           for _ in range(5)))
    return {"tool": status_tool.name, "wall_s": round(time.perf_counter() - start, 2)}

table([await five_runs(t) for t in (status_blocking, status_async, status_sync)], ["tool", "wall_s"])

**You should now see** about 1.5 s for `status_blocking`, and about 0.3 s for `status_async` and `status_sync`.

### What just happened?

The blocking tool serialised five runs, and in a server it would also freeze every other request on that worker, including health checks. A plain `def` tool is safe: LangChain runs sync tools in a thread pool under `ainvoke`. The dangerous combination is `async def` with blocking code inside. Find it with asyncio's debug mode (`PYTHONASYNCIODEBUG=1` logs callbacks slower than 100 ms) or by load testing (Part 4).

---
## Part 3 · Where sync code runs: the thread pool, and its size

Sync nodes and sync tools under `ainvoke`/`astream` run in a worker thread pool. Python's default pool has `min(32, os.cpu_count() + 4)` threads, so on an 8-core laptop at most 12 sync calls run at once, however many runs you start. A graph whose nodes are sync functions inherits that ceiling.

In [ ]:
# 🧪 Offline: 40 runs of a graph whose only node is SYNC and waits 0.2 s, vs the same node as async.
import os
from typing import TypedDict

from langgraph.graph import END, START, StateGraph

class S(TypedDict):
    n: int

def sync_node(state: S) -> S:
    time.sleep(0.2)
    return {"n": state["n"] + 1}

async def async_node(state: S) -> S:
    await asyncio.sleep(0.2)
    return {"n": state["n"] + 1}

def graph_of(node):
    return StateGraph(S).add_node("work", node).add_edge(START, "work").add_edge("work", END).compile()

async def forty(graph, label):
    start = time.perf_counter()
    await asyncio.gather(*(graph.ainvoke({"n": i}) for i in range(40)))
    return {"node": label, "wall_s": round(time.perf_counter() - start, 2)}

print("default thread pool size:", min(32, (os.cpu_count() or 1) + 4))
table([await forty(graph_of(sync_node), "sync"), await forty(graph_of(async_node), "async")], ["node", "wall_s"])

**You should now see** your pool size, then the async node at about 0.2 s and the sync node slower: about 40 ÷ pool size × 0.2 s (0.8 s with 12 threads).

### What just happened?

Async nodes overlap freely; sync nodes are capped by the pool. That's why the support desk's handoff graph (`mastery.agents.handoff_graph`) gives each node an async twin: under `astream` the specialists call `agent.ainvoke`, so a slow model call waits on the event loop instead of holding a thread for the whole turn. The model clients matter too: `ChatGroq` has native async methods, so `ainvoke` never needs a thread.

---
## Part 4 · Many runs, a limit, and a queue

A service must not start every request at once: the provider has a rate limit, memory is finite, and 200 simultaneous runs make all of them slow. `mastery.service.RunManager` runs each request as a background task behind a **semaphore** (`max_concurrent`) with a **bounded queue** (`max_queue`). Each run records its queue time, so the latency budget of 12.01 is measured, not guessed.

We run 20 requests through the support desk with a **simulated** model latency of 0.5 s per model call (`delay_s`: the offline model waits with `asyncio.sleep`; real Groq calls take a similar time). The question needs one model call, so each run takes about 0.5 s.

In [ ]:
# 🧪 Offline: 20 desk runs at different concurrency limits (simulated model latency).
from mastery.service import percentile
from mastery.service.desk import support_manager
from mastery.service.settings import ServiceSettings

QUESTION = {"messages": [{"role": "user", "content": "Hi, I'm C-1002. What plan am I on?"}]}

async def burst(limit, n=20, queue=100):
    manager = support_manager(ServiceSettings(max_concurrent=limit, max_queue=queue), delay_s=0.5)
    start = time.perf_counter()
    runs = []
    for i in range(n):
        try:
            runs.append(await manager.submit(thread_id=f"t{i}", user_id="C-1002", input=QUESTION))
        except Exception as exc:
            runs.append(type(exc).__name__)
    for run in runs:
        if not isinstance(run, str):
            await manager.wait(run.run_id)
    records = manager.records()
    return {"max_concurrent": limit, "max_queue": queue, "accepted": len(records),
            "rejected": sum(isinstance(r, str) for r in runs), "wall_s": round(time.perf_counter() - start, 2),
            "queue_p95_s": round(percentile([r.stages["queue"] for r in records], 95), 2),
            "ttft_p95_s": round(percentile([r.ttft_s for r in records], 95), 2)}

rows = [await burst(limit) for limit in (1, 4, 20)]
rows.append(await burst(4, queue=8))
table(rows, ["max_concurrent", "max_queue", "accepted", "rejected", "wall_s", "queue_p95_s", "ttft_p95_s"])

**You should now see** queue p95 falling from about 9 s (one at a time) to about 2 s (limit 4) and 0 s (limit 20), with TTFT p95 about half a second above it and wall time from about 10 s to 0.6 s; the last row accepts 12 and rejects 8 (`ServiceBusy`: 4 running + 8 queued).

### What just happened?

- This is Lesson 12.01's TTFT problem in miniature: the model didn't get slower, requests waited for a slot. Queue time belongs in the latency budget.
- A higher limit fixes queue time **only if** the provider and the machine can take it (Part 6).
- A bounded queue is **back-pressure**: past it, the service answers 429 at once (Lesson 12.04) instead of letting every request time out. Rejecting fast is kinder than accepting and failing slowly.

---
## Part 5 · Timeouts and cancellation

Every run needs a deadline (`run_timeout_s`), and users need a stop button. Both work through **task cancellation**: Python raises `CancelledError` inside the task at its current `await`, so `finally` blocks run and nothing is left half-registered. Two things to remember:

- Cancellation only lands at an `await`. Blocking code (Part 1) can't be cancelled until it returns.
- Work already done stays done. If the refund tool ran before the cancel, the refund happened: side effects need idempotency (Lesson 12.04), and the run should say what it did.

In [ ]:
# 🧪 Offline: a deadline, a cancel while running, and a cancel while queued.
manager = support_manager(ServiceSettings(max_concurrent=1, run_timeout_s=0.8), delay_s=1.0)
slow_run = await manager.submit(thread_id="a", user_id="C-1002", input=QUESTION)
queued = await manager.submit(thread_id="b", user_id="C-1002", input=QUESTION)
await manager.wait(slow_run.run_id)                    # hits the 0.8 s deadline
await asyncio.sleep(0.2)
running = queued                                       # now it has the slot, mid model call
await manager.cancel(running.run_id)
waiting = await manager.submit(thread_id="c", user_id="C-1002", input=QUESTION)
blocker = await manager.submit(thread_id="d", user_id="C-1002", input=QUESTION)
await manager.cancel(blocker.run_id)                   # cancelled before it ever started
await manager.cancel(waiting.run_id)
for run in (slow_run, running, waiting, blocker):
    print(f"{run.thread_id}: {run.status:<9} events={[e.event for e in run.events]} error={run.error}")
print(manager.stats())

**You should now see** thread `a` ending `timeout` with `events=['metadata', 'error']` and `error=run exceeded 0.8 s` (the simulated model call takes 1 s), `b`, `c` and `d` `cancelled` with only `metadata` and `end`, and `running: 0, queued: 0` in the stats.

### What just happened?

Every path ends in a terminal status with an event, so a client following the stream always learns how the run ended. The checkpointer keeps the thread's last completed step, so a later message on thread `b` continues from a consistent state.

---
## Part 6 · How much concurrency can the provider take?

**Little's law**: average runs in progress = arrival rate × average run time. Turn it around to size the limit from the provider's quota. A desk run makes about 3 model calls of ~1.5 s each, with ~2,000 tokens per call.

In [ ]:
# 🧪 Offline: what the Groq free plan allows for gpt-oss-20b (30 RPM, 8K TPM, 1K RPD, 200K TPD; Module 11 README, 2026-10-15).
calls_per_run, seconds_per_call, tokens_per_call = 3, 1.5, 2_000
limits = {"requests/min": 30, "tokens/min": 8_000, "requests/day": 1_000, "tokens/day": 200_000}

runs_per_min = min(limits["requests/min"] / calls_per_run, limits["tokens/min"] / (calls_per_run * tokens_per_call))
runs_per_day = min(limits["requests/day"] / calls_per_run, limits["tokens/day"] / (calls_per_run * tokens_per_call))
run_time_s = calls_per_run * seconds_per_call
in_progress = runs_per_min / 60 * run_time_s
print(f"runs per minute: {runs_per_min:.1f} (tokens/min is the binding limit)")
print(f"runs per day:    {runs_per_day:.0f}")
print(f"average runs in progress at that rate (Little's law): {in_progress:.2f}")

**You should now see** about 1.3 runs per minute, 33 runs per day, and 0.1 runs in progress on average.

### What just happened?

On the free plan the provider, not your code, is the bottleneck: a concurrency limit above 2–3 only fills the queue faster, and 20 simultaneous LIVE users would mostly get 429s. That's why the module's load tests run in DEMO mode with simulated latency, and why production needs a paid tier sized with this same arithmetic (runs per minute × tokens per run, plus headroom for retries).

---
## Part 7 · 💸 Concurrent calls on the real model

Three short questions, sequential vs `abatch` with `max_concurrency=3`. The rate limiter keeps us within the free plan's requests per minute.

> 💸 Calls `gpt-oss-20b` (a few calls, a few thousand tokens of your free-tier quota). Without `GROQ_API_KEY` it prints `⏭️ Skipped`.

In [ ]:
if HAVE_GROQ_KEY:
    from langchain_core.rate_limiters import InMemoryRateLimiter

    from mastery.models import get_model

    live = get_model("fast", max_tokens=120,
                     rate_limiter=InMemoryRateLimiter(requests_per_second=2, check_every_n_seconds=0.05, max_bucket_size=3))
    questions = ["Define latency in one sentence.", "Define throughput in one sentence.", "Define back-pressure in one sentence."]
    start = time.perf_counter()
    for q in questions:
        await live.ainvoke(q)
    sequential = time.perf_counter() - start
    start = time.perf_counter()
    await live.abatch(questions, config={"max_concurrency": 3})
    concurrent = time.perf_counter() - start
    print(f"sequential {sequential:.2f} s · concurrent {concurrent:.2f} s")
else:
    print("⏭️ Skipped: GROQ_API_KEY is not set (see the setup cell).")

**You should now see** the concurrent version taking about as long as the slowest single call, typically under half the sequential time (≈1K tokens in total).

### What just happened?

`abatch` with `max_concurrency` is the one-line version of Part 4's semaphore for a known list of inputs. In a service, requests arrive one by one, so the limit lives in the run manager instead.

---
## ⚠️ Pitfalls and troubleshooting

| Symptom | Cause | Fix |
|---|---|---|
| Health check times out under load | Blocking call inside `async def` | `await` an async client, or `asyncio.to_thread` |
| More workers, no faster | Sync nodes capped by the thread pool, or the provider's rate limit | Async nodes and clients; size with Little's law |
| `RuntimeError: asyncio.run() cannot be called from a running event loop` | `asyncio.run` inside Jupyter | Use `await` directly in the cell |
| Cancel doesn't stop the run | Code stuck in a blocking call | Cancellation lands at the next `await`; remove blocking code |
| Refund issued, run says "cancelled" | Cancelled after the tool ran | Idempotency keys; report completed side effects |
| Every request slow at peak | No limit: all runs share the provider quota and the CPU | `max_concurrent` + bounded queue + 429 |

## ⚠️ Production notes

> ⚠️ **Production note: one process is one event loop.**
> Uvicorn with several workers (or several containers) multiplies the loop, the semaphore and the in-memory state. The concurrency limit, the queue and the rate limiter become per-process; global limits need shared state (Redis, Postgres) and the runs themselves need a shared store (Lesson 12.05).

> ⚠️ **Production note: queues outside the web process.** (Part 4)
> For long or bursty workloads, put runs on a durable queue with separate workers, so a deploy or a crash doesn't drop queued work. LangSmith Deployment's Agent Server does this with Postgres and Redis (Lesson 12.08).

## 🏋️ Exercises

1. **(Easy)** In Part 1, replace `asyncio.gather` with `asyncio.TaskGroup`. What changes when one task raises?
2. **(Medium)** Add a fifth row to Part 4: limit 8 with `delay_s=1.0`. Predict queue p95 before running it.
3. **(Medium)** Write `with_deadline(coro, seconds)` using `asyncio.timeout` that returns `None` on timeout, and use it around one desk run.
4. **(Stretch)** Turn on asyncio debug mode (`asyncio.get_running_loop().set_debug(True)`, `slow_callback_duration = 0.05`) and catch `status_blocking` from Part 2 in the log.

<details>
<summary><b>Show solutions</b></summary>

**1.** `TaskGroup` cancels the remaining tasks when one fails and raises an `ExceptionGroup`; `gather` (without `return_exceptions=True`) raises the first error but lets the others keep running. In a server, prefer `TaskGroup` for work that belongs together.

**2.** 20 runs, 8 at a time, ~1 s per run (one model call): three waves (8, 8, 4), so the last runs wait about 2 s. Queue p95 ≈ 2 s, TTFT p95 ≈ 3 s. You'll need `burst` to accept a `delay_s` argument.

**3.**
```python
async def with_deadline(coro, seconds):
    try:
        async with asyncio.timeout(seconds):
            return await coro
    except TimeoutError:
        return None
from mastery.service.desk import support_graph
graph, desk = support_graph("demo", delay_s=0.5)
print(await with_deadline(graph.ainvoke(QUESTION, {"configurable": {"thread_id": "x"}}), 0.3))   # None
```

**4.**
```python
import logging
logging.basicConfig(level=logging.WARNING)
loop = asyncio.get_running_loop()
loop.set_debug(True); loop.slow_callback_duration = 0.05
await five_runs(status_blocking)        # "Executing <Task ...> took 0.30 seconds"
loop.set_debug(False)
```
</details>

## 🔎 Docs drill (5 min)

In FastAPI, what happens when you declare an endpoint with plain `def` instead of `async def`, and why doesn't it block the server?

Try it before you open the answer: use the official docs, the API reference, or the installed source (`??` in Jupyter).

<details>
<summary>Path to the answer</summary>

1. Open [Concurrency and async / await](https://fastapi.tiangolo.com/async/) and find the section on path operation functions.
2. A plain `def` endpoint runs in an external **thread pool** and is awaited, so it doesn't block the event loop.
3. Connect it to Part 3: that pool has a size, so many slow `def` endpoints still queue.

</details>

## ✅ Checkpoint

- [ ] I can show the difference between blocking and awaiting in a few lines.
- [ ] I found and fixed a blocking tool inside an async agent.
- [ ] I know sync code under `ainvoke` runs in a thread pool, and that pool's size.
- [ ] I measured queue time and TTFT against a concurrency limit, and saw back-pressure reject runs.
- [ ] I used a deadline and cancelled runs, and every run ended with a terminal status.
- [ ] I sized concurrency from a provider quota with Little's law.
- [ ] Restart & Run All passes; outputs cleared before saving.

## 🔎 Research checkpoint

Check the [FastAPI release notes](https://fastapi.tiangolo.com/release-notes/) and log in [`docs/research-log.md`](../../docs/research-log.md) whether the `opentelemetry-api>=1.44` requirement of fastapi 0.142+ still conflicts with `langgraph-api` (2026-10-16: it does, so the course env pins fastapi below 0.142; see `pyproject.toml`).

## Summary

- Async helps only if every wait is an `await`; blocking code freezes the loop.
- Sync code runs in a thread pool whose size caps concurrency; prefer async nodes and clients.
- Limit concurrency, bound the queue, reject fast (429), and measure queue time.
- Deadlines and cancellation go through `CancelledError`; completed side effects stay done.
- Little's law sizes concurrency from the provider's quota; on the free plan the quota is the bottleneck.

---
## ➡️ What's next

**You're done with this lesson when** every box in the ✅ Checkpoint is ticked and **Kernel → Restart Kernel and Run All Cells…** finishes without errors.

1. Clear outputs before you save (**Edit → Clear Outputs of All Cells**), then **File → Save** (`Ctrl + S`).
2. Optional: log your research checkpoint in [`docs/research-log.md`](../../docs/research-log.md).
3. Next: **Lesson 12-04-fastapi-service.ipynb** → [`12-04-fastapi-service.ipynb`](12-04-fastapi-service.ipynb).

[Module 12 README](README.md) · [Course outline](../../COURSE_OUTLINE.md)